# Chapter 6 &mdash; Minimization as a Fixed-Point Computation

**Concept 10 of the Chapter 6 decomposition:** *Minimization as a Fixed-Point Computation: `fixptDist`*

Apply the propagation rule until the table stops changing &mdash; a fixed point, reached monotonically.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6/Concept-Fixed-Point-Minimization/Concept-Fixed-Point-Minimization.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


`fixptDist` is the whole algorithm in one idea: **apply a rule repeatedly until
nothing changes**.

The rule only ever turns $-1$ into a number, never the other way, so the table is
**monotone**; the table is **finite**; therefore a **fixed point** exists and is
reached in finitely many rounds &mdash; at most $|Q|-1$ of them.

This is the same pattern as dataflow analysis in compilers and as the closure
computations in Chapter 11. Recognising it saves you from re-deriving termination
arguments.

## 2. Definitions

### A chain machine, which needs the maximum number of rounds

In [ ]:
chain = md2mc('''DFA
I  : 0 -> A
A  : 0 -> B
B  : 0 -> C
C  : 0 -> F
F  : 0 -> F
I  : 1 -> I
A  : 1 -> A
B  : 1 -> B
C  : 1 -> C
F  : 1 -> F
''')

### The fixed-point loop, instrumented to count rounds

In [ ]:
def fixpoint(D):
    qs = sorted(D["Q"])
    pairs = [(a, b) for i, a in enumerate(qs) for b in qs[i+1:]]
    dist = {p: (0 if (p[0] in D["F"]) != (p[1] in D["F"]) else -1) for p in pairs}
    def key(x, y): return (x, y) if (x, y) in dist else (y, x)
    rounds, k = 0, 0
    while True:
        changed = False
        for (p, q) in pairs:
            if dist[(p, q)] != -1: continue
            for a in sorted(D["Sigma"]):
                s, t = step_dfa(D, p, a), step_dfa(D, q, a)
                if s != t and dist[key(s, t)] != -1 and dist[key(s, t)] <= k:
                    dist[(p, q)] = k + 1; changed = True; break
        rounds += 1; k += 1
        if not changed: return dist, rounds

## 3. Tests

Monotonicity: marks are only ever added, never removed.

In [ ]:
dist, rounds = fixpoint(chain)
print("rounds to fixpoint :", rounds, "   |Q|-1 =", len(chain["Q"]) - 1)
print("distances :", dict(sorted(dist.items())))
assert rounds <= len(chain["Q"])

Every pair in this chain is distinguishable, so nothing merges.

In [ ]:
print("pairs still at -1 :", [k for k, v in dist.items() if v == -1])
m = min_dfa(chain)
print("|Q| before %d, after %d" % (len(chain["Q"]), len(m["Q"])))
assert len(m["Q"]) == len(chain["Q"])
print("already minimal.")

A mergeable machine reaches its fixed point in fewer rounds and shrinks.

In [ ]:
redundant = md2mc('''DFA
I  : 0 -> A
I  : 1 -> B
A  : 0 | 1 -> F
B  : 0 | 1 -> F
F  : 0 | 1 -> F
''')
d2, r2 = fixpoint(redundant)
print("rounds :", r2, "  equivalent pairs :", [k for k, v in d2.items() if v == -1])
print("|Q| %d -> %d" % (len(redundant["Q"]), len(min_dfa(redundant)["Q"])))
assert len(min_dfa(redundant)["Q"]) < len(redundant["Q"])

Idempotence: minimizing a minimal machine changes nothing. That *is* the fixed point.

In [ ]:
m = min_dfa(redundant)
mm = min_dfa(m)
print("min(min(D)) isomorphic to min(D)?", iso_dfa(m, mm))
assert iso_dfa(m, mm) and len(m["Q"]) == len(mm["Q"])

## 4. Exercises


1. Why is the bound $|Q|-1$ and not $|Q|$?
2. Name two other fixed-point computations in this book.
3. Construct a DFA needing exactly 5 rounds.

In [ ]:
# Your work for the exercises above.